# **Week 7: Interpolasi**

* Nama  : Alfian Daffa Baihaqi
* NIU   : 25/570509/PTK/16981
* Prodi : Magister Teknik Elektro

In [4]:
import numpy as np

# Data points
x = np.array([0, 1, 2, 3, 4], dtype=float)
y = np.array([1, 3, 2, 5, 4], dtype=float)

## **1. Divided Difference**
Newton Divided Difference Algorithm
Given data points: $ (x_0,y_0),(x_1,y_1),\dots,(x_n,y_n) $
1. Initialize the first column of the divided difference table: $ f[x_i] = y_i $

2. Compute higher-order divided differences:
$$
f[x_i,x_{i+1},\dots,x_{i+j}] =
\frac{f[x_{i+1},\dots,x_{i+j}] - f[x_i,\dots,x_{i+j-1}]}
{x_{i+j}-x_i}
$$

3. The Newton interpolation polynomial is:

$$
P(x)=a_0 + a_1(x-x_0) + a_2(x-x_0)(x-x_1) + \dots + a_n(x-x_0)\cdots(x-x_{n-1})
$$

where $a_i = f[x_0,x_1,\dots,x_i]$

In [5]:
def divided_difference(x, y):
    n = len(x)
    table = np.zeros((n, n+1))
    table[:,0] = x
    table[:,1] = y
    # Hitung Kolom Selanjutnya
    for j in range(2, n+1):
        for i in range(n-(j-1)):
            table[i,j] = (table[i+1,j-1] - table[i,j-1]) / (x[i+(j-1)] - x[i])
            print(f"table[{i},{j}] = ({table[i+1,j-1]:.2f} - {table[i,j-1]:.2f}) / ({x[i+(j-1)]:.2f} - {x[i]:.2f}) = {table[i,j]:.2f}")
    # Cetak Tabel
    print("\nNewton Divided Difference Table:")
    header = ["x", "f(x)"] + [f"f[{' '.join(['x']*k)}]" for k in range(1, n)]
    print(" ".join(f"{h:>10}" for h in header))
    for i in range(n):
        for j in range(n+1):
            if j <= n-i:
                print(f"{table[i,j]:10.4f}", end=" ")
        print()

    return table[0,1:]


def newton_predict(x, coefficients, x_val):
    n = len(coefficients)
    result = coefficients[0]
    product = 1
    for i in range(1, n):
        product *= (x_val - x[i-1])
        result += coefficients[i] * product
    return result


In [6]:
coefficients = divided_difference(x, y)

print("\nNewton Coefficients:")
for i, coeff in enumerate(coefficients):
    print(f"a{i} = {coeff:.4f}")

print("\nInterpolation Polynomial:")
terms = []
for i in range(len(coefficients)): 
    term = f"{coefficients[i]:.4f}"
    for j in range(i):
        term += f"*(x - {x[j]:.2f})"
    terms.append(term)
polynomial = " + ".join(terms)
print(f"P(x) = {polynomial}")  

x_test = 2.55
prediction = newton_predict(x, coefficients, x_test)
print(f"\nPrediction:")
print(f"P({x_test}) = {prediction:.4f}")

table[0,2] = (3.00 - 1.00) / (1.00 - 0.00) = 2.00
table[1,2] = (2.00 - 3.00) / (2.00 - 1.00) = -1.00
table[2,2] = (5.00 - 2.00) / (3.00 - 2.00) = 3.00
table[3,2] = (4.00 - 5.00) / (4.00 - 3.00) = -1.00
table[0,3] = (-1.00 - 2.00) / (2.00 - 0.00) = -1.50
table[1,3] = (3.00 - -1.00) / (3.00 - 1.00) = 2.00
table[2,3] = (-1.00 - 3.00) / (4.00 - 2.00) = -2.00
table[0,4] = (2.00 - -1.50) / (3.00 - 0.00) = 1.17
table[1,4] = (-2.00 - 2.00) / (4.00 - 1.00) = -1.33
table[0,5] = (-1.33 - 1.17) / (4.00 - 0.00) = -0.62

Newton Divided Difference Table:
         x       f(x)       f[x]     f[x x]   f[x x x] f[x x x x]
    0.0000     1.0000     2.0000    -1.5000     1.1667    -0.6250 
    1.0000     3.0000    -1.0000     2.0000    -1.3333 
    2.0000     2.0000     3.0000    -2.0000 
    3.0000     5.0000    -1.0000 
    4.0000     4.0000 

Newton Coefficients:
a0 = 1.0000
a1 = 2.0000
a2 = -1.5000
a3 = 1.1667
a4 = -0.6250

Interpolation Polynomial:
P(x) = 1.0000 + 2.0000*(x - 0.00) + -1.5000*(x - 0.0

## **2. Lagrange Polynomial**

Given data points:$ (x_0,y_0),(x_1,y_1),\dots,(x_n,y_n) $

1. Construct the Lagrange basis polynomial:
$
L_i(x) =
\prod_{\substack{j=0 \\ j \ne i}}^{n}
\frac{x-x_j}{x_i-x_j}
$

2. Multiply each basis polynomial by its corresponding value: $ y_i L_i(x) $

3. Sum all terms to obtain the interpolation polynomial:
$ P(x)=\sum_{i=0}^{n} y_i L_i(x) $

In [7]:
import numpy as np

def lagrange_polynomial(x, y, x_eval=None):
    n = len(x)

    print("\nLagrange Basis Polynomials:")
    # Cari L_i(x) untuk setiap i
    for i in range(n):
        numerator = ""
        denominator = 1
        for j in range(n):
            if i != j:
                numerator += f"(x - {x[j]})"
                denominator *= (x[i] - x[j])

        print(f"L_{i}(x) = {numerator} / {denominator}")

    # Gabungkan untuk membentuk polinomial interpolasi
    print("\nInterpolation Polynomial:")
    poly = "P(x) = "
    for i in range(n):
        numerator = ""
        denominator = 1
        for j in range(n):
            if i != j:
                numerator += f"(x - {x[j]})"
                denominator *= (x[i] - x[j])
        term = f"{y[i]} * ({numerator}) / {denominator}"
        if i == 0:
            poly += term
        else:
            poly += " + " + term
    print(poly)

    # Prediksi nilai pada x tertentu
    if x_eval is not None:
        result = 0.0
        for i in range(n):
            term = y[i]
            for j in range(n):
                if i != j:
                    term *= (x_eval - x[j]) / (x[i] - x[j])
            result += term
        print(f"\nPrediction:")
        print(f"P({x_eval}) = {result:.4f}")



In [8]:
lagrange_polynomial(x, y, x_eval=2.55)


Lagrange Basis Polynomials:
L_0(x) = (x - 1.0)(x - 2.0)(x - 3.0)(x - 4.0) / 24.0
L_1(x) = (x - 0.0)(x - 2.0)(x - 3.0)(x - 4.0) / -6.0
L_2(x) = (x - 0.0)(x - 1.0)(x - 3.0)(x - 4.0) / 4.0
L_3(x) = (x - 0.0)(x - 1.0)(x - 2.0)(x - 4.0) / -6.0
L_4(x) = (x - 0.0)(x - 1.0)(x - 2.0)(x - 3.0) / 24.0

Interpolation Polynomial:
P(x) = 1.0 * ((x - 1.0)(x - 2.0)(x - 3.0)(x - 4.0)) / 24.0 + 3.0 * ((x - 0.0)(x - 2.0)(x - 3.0)(x - 4.0)) / -6.0 + 2.0 * ((x - 0.0)(x - 1.0)(x - 3.0)(x - 4.0)) / 4.0 + 5.0 * ((x - 0.0)(x - 1.0)(x - 2.0)(x - 4.0)) / -6.0 + 4.0 * ((x - 0.0)(x - 1.0)(x - 2.0)(x - 3.0)) / 24.0

Prediction:
P(2.55) = 3.3188


## **3. Analisis**

Hasil prediksi menggunakan metode Newton Divided Difference dan Lagrange Interpolation pada titik ( P(2.55) ) menghasilkan nilai yang sama, yaitu 3.3188. Kesamaan hasil ini terjadi karena kedua metode tersebut membangun polinomial interpolasi yang sama dari himpunan titik data yang sama, meskipun menggunakan pendekatan perhitungan yang berbeda.